In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp

In [2]:
data = pd.read_csv('Total_data.csv')

data

/var/folders/zr/rdm09ntx305728qxtml43fk40000gn/T/ipykernel_27210/2123774900.py:1: DtypeWarning: Columns (0: assay_category, 1: assay_strain, 2: assay_tissue, 3: assay_cell_type, 4: assay_subcellular_fraction) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('Total_data.csv')


,Unnamed: 0,assay_id,description,doc_id,year,confidence_score,targed_id,pref_name,protein_sequence,uniprot_accession,...,pchembl_value,assay_type,assay_organism,assay_category,assay_tax_id,assay_strain,assay_tissue,assay_cell_type,assay_subcellular_fraction,bao_format
0,0,CHEMBL872937,In vivo inhibitory activity against human Hepa...,CHEMBL1146658,2004.0,8,CHEMBL3921,Heparanase,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,Q9Y251,...,5.60,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000218
1,1,CHEMBL872937,In vivo inhibitory activity against human Hepa...,CHEMBL1146658,2004.0,8,CHEMBL3921,Heparanase,MLLRSKPALPPPLMLLLLGPLGPLSPGALPRPAQAQDVVDLDFFTQ...,Q9Y251,...,5.05,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000218
2,2,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,8,CHEMBL4632,Peroxisomal acyl-coenzyme A oxidase 1,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,...,5.40,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
3,3,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,8,CHEMBL4632,Peroxisomal acyl-coenzyme A oxidase 1,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,...,4.77,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
4,4,CHEMBL760688,Inhibitory activity against Palmitoyl-CoA oxid...,CHEMBL1148425,2004.0,8,CHEMBL4632,Peroxisomal acyl-coenzyme A oxidase 1,MNPDLRKERASATFNPELITHILDGSPENTRRRREIENLILNDPDF...,P07872,...,6.75,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BAO_0000357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1414959,1414959,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,...,9.52,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
1414960,1414960,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,...,6.13,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
1414961,1414961,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,...,7.14,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357
1414962,1414962,CHEMBL5739541,"IRAK4 Enzymatic DELFIA Assay, Protocol A: This...",CHEMBL5729811,2023.0,9,CHEMBL3778,Interleukin-1 receptor-associated kinase 4,MNKPITPSTYVRCLNVGLIRKLSDFIDPQEGWKKLAVAIKKPSGDD...,Q9NWZ3,...,7.23,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000357


In [3]:
#Removing data with confidence score lower than 9

def remove_low_confidence(data):
    
    mask = (data['confidence_score'] == 9)

    df =  data[mask]

    return df

#Removing data with missing documentation (no document year)

def remove_missing_doc_year(data):

    mask = pd.isna(data['year'])

    df = data[~mask]

    return df

# Filter by assay size

def filter_by_size(data, minAssaySize, maxAssaySize):

    #Count how many times each assay appears in filtered_activities
    count = data['assay_id'].value_counts()

    valid_ids = count[(count >= minAssaySize) & (count <= maxAssaySize)].index

    df = data[data['assay_id'].isin(valid_ids)]

    return df

#Removing mutated protein

def remove_mutant(data):
    
    mask = pd.isna(data['variant_id'])

    data = data[mask]

    exclude_keywords = ['mutant', 'mutation', 'variant']

    pattern = '|'.join(exclude_keywords)

    df = data[~data['description'].str.contains(pattern, case=False, na=False)]

    return df

# Removing experiments with different pChembl value

def filter_pChembl_value(data): #Verify this function (made by Claude)

    pairs = data.merge(
        data,
        on=['molecule_id', 'targed_id'],  # same molecule and target
        suffixes=('_1', '_2')
    )

    pairs = pairs[pairs['assay_id_1'] != pairs['assay_id_2']]

    pairs = pairs[pairs['assay_id_1'] < pairs['assay_id_2']]

    pairs['pchembl_diff'] = abs(pairs['pchembl_value_1'] - pairs['pchembl_value_2'])

    pairs_filtered = pairs[~((pairs['pchembl_diff'] == 0) | (pairs['pchembl_diff'] == 3.0))]

    valid_assays_1 = pairs_filtered[['molecule_id', 'targed_id', 'assay_id_1']].rename(
        columns={'assay_id_1': 'assay_id'}
    )

    valid_assays_2 = pairs_filtered[['molecule_id', 'targed_id', 'assay_id_2']].rename(
        columns={'assay_id_2': 'assay_id'}
    )

    valid_measurements = pd.concat([valid_assays_1, valid_assays_2]).drop_duplicates()

    df_filtered = data.merge(
        valid_measurements,
        on=['molecule_id', 'targed_id', 'assay_id'],
        how='inner'  # Keep only rows that match
    )

    return df_filtered

# Removing assays with different assay_type

def removing_different_assay_type(data):

    inconsistent_ids = data.groupby('assay_id')['assay_type'].nunique()
    inconsistent_ids = inconsistent_ids[inconsistent_ids > 1].index

    df_clean = data[~data['assay_id'].isin(inconsistent_ids)]

    return df_clean

# Removing assays with different assay metadata

# Removing duplicate papers

def removing_duplicate_paper(data):

    inconsistent_ids = data.groupby(['doc_id', 'targed_id'])['assay_id'].nunique()
    inconsistent_ids = inconsistent_ids[inconsistent_ids > 1].index

    df_clean = data[~data['assay_id'].isin(inconsistent_ids)]

    return df_clean


In [4]:
#Test function

#Data = filter_by_size(data, 20, 100)
#Data = filter_pChembl_value(Data)

#Data = removing_duplicate_paper(Data)
Data = remove_low_confidence(data)
Data = remove_missing_doc_year(Data)
Data = remove_mutant(Data)
Data = filter_by_size(Data, 20, 100)


#df = Data.value_counts('assay_id')

Data

,Unnamed: 0,assay_id,description,doc_id,year,confidence_score,targed_id,pref_name,protein_sequence,uniprot_accession,...,pchembl_value,assay_type,assay_organism,assay_category,assay_tax_id,assay_strain,assay_tissue,assay_cell_type,assay_subcellular_fraction,bao_format
20,20,CHEMBL641889,Inhibition of Acyl coenzyme A:cholesterol acyl...,CHEMBL1128039,1995.0,9,CHEMBL4464,Sterol O-acyltransferase 1,MSLRNRLSKSGENPEQDEAQKNFMDTYRNGHITMKQLIAKKRLLAA...,Q61263,...,5.26,B,Mus musculus,NaN,10090.0,NaN,NaN,Macrophage,NaN,BAO_0000219
22,22,CHEMBL641889,Inhibition of Acyl coenzyme A:cholesterol acyl...,CHEMBL1128039,1995.0,9,CHEMBL4464,Sterol O-acyltransferase 1,MSLRNRLSKSGENPEQDEAQKNFMDTYRNGHITMKQLIAKKRLLAA...,Q61263,...,6.35,B,Mus musculus,NaN,10090.0,NaN,NaN,Macrophage,NaN,BAO_0000219
68,68,CHEMBL645934,Inhibitory activity against acyl coenzyme A:ch...,CHEMBL1127563,1994.0,9,CHEMBL285,Sterol O-acyltransferase 1,MVGEETSLRNRLSRSAENPEQDEAQKNLLDTHRNGHITMKQLIAKK...,O70536,...,7.82,B,Rattus norvegicus,NaN,10116.0,NaN,NaN,NaN,Microsome,BAO_0000251
74,74,CHEMBL769366,In vivo antiviral activity (IC50) against HIV-...,CHEMBL1129270,1996.0,9,CHEMBL243,Human immunodeficiency virus type 1 protease,PQVTLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKM...,Q72874,...,7.38,B,Human immunodeficiency virus 1,NaN,11676.0,NaN,NaN,NaN,NaN,BAO_0000218
88,88,CHEMBL701719,"Inhibition of HIV-1 integrase, under 1 uM for ...",CHEMBL1133723,2000.0,9,CHEMBL3471,Human immunodeficiency virus type 1 integrase,FLDGIDKAQDEHEKYHSNWRAMASDFNLPPVVAKEIVASCDKCQLK...,Q7ZJM1,...,5.05,B,Human immunodeficiency virus 1,NaN,11676.0,NaN,NaN,NaN,NaN,BAO_0000357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1414381,1414381,CHEMBL5739536,HTRF Displacement Assay: Table 21: This Exampl...,CHEMBL5729810,2023.0,9,CHEMBL2079846,Son of sevenless homolog 1,MQAQQLPYEFFSEENAPKWRGLLVPALKKVQGQVHPTLESNDDALQ...,Q07889,...,8.20,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
1414382,1414382,CHEMBL5739536,HTRF Displacement Assay: Table 21: This Exampl...,CHEMBL5729810,2023.0,9,CHEMBL2079846,Son of sevenless homolog 1,MQAQQLPYEFFSEENAPKWRGLLVPALKKVQGQVHPTLESNDDALQ...,Q07889,...,8.35,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
1414383,1414383,CHEMBL5739536,HTRF Displacement Assay: Table 21: This Exampl...,CHEMBL5729810,2023.0,9,CHEMBL2079846,Son of sevenless homolog 1,MQAQQLPYEFFSEENAPKWRGLLVPALKKVQGQVHPTLESNDDALQ...,Q07889,...,8.60,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
1414384,1414384,CHEMBL5739536,HTRF Displacement Assay: Table 21: This Exampl...,CHEMBL5729810,2023.0,9,CHEMBL2079846,Son of sevenless homolog 1,MQAQQLPYEFFSEENAPKWRGLLVPALKKVQGQVHPTLESNDDALQ...,Q07889,...,8.14,B,Homo sapiens,NaN,9606.0,NaN,NaN,NaN,NaN,BAO_0000019
